<a href="https://colab.research.google.com/github/ancestor9/2026_Spring_Application_Developments/blob/main/week7/0422_sync_async_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **동기(Sync) vs 비동기(Async) 방식**

## **동기방식(Sync)**
- 코드가 파일을 **하나씩 순차적으로** 처리하므로, 앞 작업이 끝나야 다음 작업이 시작
- 파일 수가 많아지면 시간이 많이 소요

## **비동기(Async) 방식**
- **여러 작업을 동시에** 실행하여 효율성을 높임
- 특정 작업(예: 네트워크 요청)을 기다리는 동안 다른 작업을 진행
- Python에서는 `asyncio`와 `aiohttp` 라이브러리로 구현

##### <font color='red'>☕ 커피샵 비유: 친구에게 커피를 부탁할 때,
- **동기(sync)**: 친구 옆에 서서 커피 받을 때까지 기다린다
- **비동기(async)**: 친구가 커피 가져오는 동안 나는 다른 일을 한다</font>

<img src='https://images.unsplash.com/photo-1516367519848-ee5b4deb59d2?w=500&auto=format&fit=crop&q=60'>

## **0. 공통 설정**
아래 셀은 동기/비동기 예제에서 공통으로 사용하는 라이브러리와 데이터를 준비합니다.

In [1]:
import os
import time
import requests
from bs4 import BeautifulSoup

# ── 다운로드할 CSV 파일 목록을 웹에서 자동으로 가져오기 ──
def get_all_csv_urls(base_url):
    """주어진 base_url 페이지에서 .csv 링크를 모두 수집하여 반환"""
    response = requests.get(base_url)
    if response.status_code != 200:
        return []
    soup = BeautifulSoup(response.text, 'html.parser')
    return [
        base_url + link.get('href')
        for link in soup.find_all('a')
        if link.get('href', '').endswith('.csv')
    ]

BASE_URL = "https://people.sc.fsu.edu/~jburkardt/data/csv/"
file_urls = get_all_csv_urls(BASE_URL)

print(f"총 {len(file_urls)}개의 CSV 파일 URL을 찾았습니다.")
for url in file_urls:
    print(" -", url)

총 33개의 CSV 파일 URL을 찾았습니다.
 - https://people.sc.fsu.edu/~jburkardt/data/csv/addresses.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/biostats.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/cities.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/crash_catalonia.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/deniro.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/example.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/faithful.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/ford_escort.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/freshman_kgs.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/freshman_lbs.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/grades.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/homes.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/hooke.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/hurricanes.csv
 - https://people.sc.fsu.edu/~jburk

### 실습을 위해 다운로드 받은 디렉토리의 데이터를 모두 삭제하는 사용자정의 함수

In [2]:
def clean_csv_files():
    """현재 디렉토리의 모든 .csv 파일을 삭제 (실습 후 정리용)"""
    deleted = []
    for file_name in os.listdir('./'):
        if file_name.endswith('.csv'):
            file_path = os.path.join('./', file_name)
            try:
                os.remove(file_path)
                deleted.append(file_name)
            except OSError as e:
                print(f"삭제 실패: {file_path} - {e.strerror}")
    print(f"{len(deleted)}개 CSV 파일 삭제 완료.")

---
## **1. 동기 방식으로 파일 다운로드**

파일을 **하나씩 순서대로** 다운로드합니다.  
> `[1]` 완료 → `[2]` 완료 → `[3]` 완료 → … 순으로 출력되는 것을 확인하세요.

In [3]:
def download_file_sync(url, save_path, index):
    """단일 파일을 동기 방식으로 다운로드"""
    print(f"[{index:2d}] 다운로드 시작: {os.path.basename(url)}")
    response = requests.get(url)
    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"[{index:2d}] ✅ 완료: {save_path}")
        return save_path
    else:
        print(f"[{index:2d}] ❌ 실패: {url} (상태코드: {response.status_code})")
        return None


def download_all_sync():
    """모든 파일을 동기 방식으로 순차 다운로드"""
    results = []
    for i, url in enumerate(file_urls, start=1):
        save_path = f"./{os.path.basename(url)}"
        result = download_file_sync(url, save_path, i)
        if result:
            results.append(os.path.basename(result))
    return results


# ── 실행 ──
print("=" * 50)
print("동기 방식 다운로드 시작")
print("=" * 50)
t1 = time.time()
sync_results = download_all_sync()
t2 = time.time()

print(f"\n⏱ 동기 방식 완료: {t2 - t1:.2f}초 소요")
print(f"📁 다운로드된 파일 목록 ({len(sync_results)}개):")
for idx, fname in enumerate(sync_results, start=1):
    print(f"  {idx:2d}. {fname}")

동기 방식 다운로드 시작
[ 1] 다운로드 시작: addresses.csv
[ 1] ✅ 완료: ./addresses.csv
[ 2] 다운로드 시작: airtravel.csv
[ 2] ✅ 완료: ./airtravel.csv
[ 3] 다운로드 시작: biostats.csv
[ 3] ✅ 완료: ./biostats.csv
[ 4] 다운로드 시작: cities.csv
[ 4] ✅ 완료: ./cities.csv
[ 5] 다운로드 시작: crash_catalonia.csv
[ 5] ✅ 완료: ./crash_catalonia.csv
[ 6] 다운로드 시작: deniro.csv
[ 6] ✅ 완료: ./deniro.csv
[ 7] 다운로드 시작: example.csv
[ 7] ✅ 완료: ./example.csv
[ 8] 다운로드 시작: faithful.csv
[ 8] ✅ 완료: ./faithful.csv
[ 9] 다운로드 시작: ford_escort.csv
[ 9] ✅ 완료: ./ford_escort.csv
[10] 다운로드 시작: freshman_kgs.csv
[10] ✅ 완료: ./freshman_kgs.csv
[11] 다운로드 시작: freshman_lbs.csv
[11] ✅ 완료: ./freshman_lbs.csv
[12] 다운로드 시작: grades.csv
[12] ✅ 완료: ./grades.csv
[13] 다운로드 시작: homes.csv
[13] ✅ 완료: ./homes.csv
[14] 다운로드 시작: hooke.csv
[14] ✅ 완료: ./hooke.csv
[15] 다운로드 시작: hurricanes.csv
[15] ✅ 완료: ./hurricanes.csv
[16] 다운로드 시작: hw_200.csv
[16] ✅ 완료: ./hw_200.csv
[17] 다운로드 시작: hw_25000.csv
[17] ✅ 완료: ./hw_25000.csv
[18] 다운로드 시작: lead_shot.csv
[18] ✅ 완료: ./lead_shot.csv
[19] 다운로드 시작: le

In [4]:
clean_csv_files()

33개 CSV 파일 삭제 완료.


---
## **2. 비동기 방식으로 파일 다운로드**

파일을 **동시에** 요청하여 다운로드합니다.  

> **핵심 관찰 포인트:**
> - `다운로드 시작` 메시지는 `[1]~[N]`이 거의 동시에 출력됨
> - `완료` 메시지의 순서는 **요청 순서와 다를 수 있음** (먼저 응답이 온 파일이 먼저 완료)
> - 전체 소요 시간이 동기보다 훨씬 짧음

- 웹에서 대량의 데이터를 아주 빠르게 긁어오거나(크롤링), 수많은 API 호출을 동시에 처리하기 위해
- 비동기 시스템(asyncio)과 비동기 전용 통신 도구(aiohttp) 모듈을 호출

In [8]:
import aiohttp
import asyncio

#### **aiohttp** : 비동기 환경에서 웹 서버와 통신(GET, POST 등)을 수행

> 기본적으로 쓰이는 requests 라이브러리는 한 번에 하나의 요청만 처리하고 응답이 올 때까지 프로그램을 멈추게(Blocking)하는 반면에 aiohttp는 수백 개의 웹 요청을 거의 동시에 보낼 수 있어 성능이 압도적으로 높음

#### **asyncio** : 이벤트 루프(Event Loop)를 생성하고 관리

> 여러 요리를 동시에 하는 주방의 **'주방장'** 으로 한 요리가 끓는 동안 가만히 기다리는 게 아니라, 다른 재료를 손질하도록 지시하는 역할

>> 주요 키워드: async def (함수 정의), await (작업 대기).

In [9]:
async def download_file_async(session, url, save_path, index):
    """단일 파일을 비동기 방식으로 다운로드"""
    print(f"[{index:2d}] 다운로드 시작: {os.path.basename(url)}")
    async with session.get(url) as response:
        if response.status == 200:
            content = await response.read()
            with open(save_path, 'wb') as f:
                f.write(content)
            print(f"[{index:2d}] ✅ 완료: {save_path}")
            return save_path
        else:
            print(f"[{index:2d}] ❌ 실패: {url} (상태코드: {response.status})")
            return None


async def download_all_async():
    """모든 파일을 비동기 방식으로 동시 다운로드"""
    async with aiohttp.ClientSession() as session:
        tasks = [
            download_file_async(session, url, f"./{os.path.basename(url)}", i)
            for i, url in enumerate(file_urls, start=1)
        ]
        # asyncio.gather: 모든 태스크를 동시에 실행하고, 완료 순서대로 결과 반환
        results = await asyncio.gather(*tasks)
    return [os.path.basename(r) for r in results if r]


# ── 실행 (Jupyter/Colab에서는 await 직접 사용, 일반 .py에서는 asyncio.run() 사용) ──
print("=" * 50)
print("비동기 방식 다운로드 시작")
print("=" * 50)
t1 = time.time()
async_results = await download_all_async()  # 일반 스크립트: asyncio.run(download_all_async())
t2 = time.time()

print(f"\n⏱ 비동기 방식 완료: {t2 - t1:.2f}초 소요")
print(f"📁 다운로드된 파일 목록 ({len(async_results)}개):")
for idx, fname in enumerate(async_results, start=1):
    print(f"  {idx:2d}. {fname}")

비동기 방식 다운로드 시작
[ 1] 다운로드 시작: addresses.csv
[ 2] 다운로드 시작: airtravel.csv
[ 3] 다운로드 시작: biostats.csv
[ 4] 다운로드 시작: cities.csv
[ 5] 다운로드 시작: crash_catalonia.csv
[ 6] 다운로드 시작: deniro.csv
[ 7] 다운로드 시작: example.csv
[ 8] 다운로드 시작: faithful.csv
[ 9] 다운로드 시작: ford_escort.csv
[10] 다운로드 시작: freshman_kgs.csv
[11] 다운로드 시작: freshman_lbs.csv
[12] 다운로드 시작: grades.csv
[13] 다운로드 시작: homes.csv
[14] 다운로드 시작: hooke.csv
[15] 다운로드 시작: hurricanes.csv
[16] 다운로드 시작: hw_200.csv
[17] 다운로드 시작: hw_25000.csv
[18] 다운로드 시작: lead_shot.csv
[19] 다운로드 시작: letter_frequency.csv
[20] 다운로드 시작: mlb_players.csv
[21] 다운로드 시작: mlb_teams_2012.csv
[22] 다운로드 시작: news_decline.csv
[23] 다운로드 시작: nile.csv
[24] 다운로드 시작: oscar_age_female.csv
[25] 다운로드 시작: oscar_age_male.csv
[26] 다운로드 시작: snakes_count_10.csv
[27] 다운로드 시작: snakes_count_100.csv
[28] 다운로드 시작: snakes_count_1000.csv
[29] 다운로드 시작: snakes_count_10000.csv
[30] 다운로드 시작: tally_cab.csv
[31] 다운로드 시작: taxables.csv
[32] 다운로드 시작: trees.csv
[33] 다운로드 시작: zillow.csv
[ 4] ✅ 완료: ./cities.csv
[

In [10]:
clean_csv_files()

33개 CSV 파일 삭제 완료.


---
## **3. 동기 vs 비동기 핵심 차이 정리**

| 항목 | 동기 (Sync) | 비동기 (Async) |
|------|------------|----------------|
| 처리 방식 | 순차 (하나씩) | 동시 (한꺼번에) |
| 완료 순서 | 항상 요청 순서와 동일 | 응답이 빠른 것부터 완료 |
| 소요 시간 | 파일 수 × 개당 시간 | ≈ 가장 느린 파일 1개의 시간 |
| 코드 복잡도 | 단순 | `async/await` 필요 |
| 적합한 상황 | 순서 보장이 중요할 때 | 네트워크 I/O가 많을 때 |

---
## **4. 비동기 핵심 개념: `async` / `await` / `asyncio`**

- **`asyncio.gather(*tasks)`**: 여러 코루틴을 동시에 실행하고 모든 결과를 모아서 반환

> - **`async def`**: 비동기 함수(코루틴) 정의. 일반 함수처럼 호출하면 실행되지 않고, `await`나 이벤트 루프를 통해야 실행됨
> - **`await`**: "이 작업이 끝날 때까지 잠깐 기다려. 그 동안 다른 작업해도 돼"라는 신호

- **`aiohttp`**: `requests`의 비동기 버전. HTTP 요청을 비동기적으로 처리

- main() 호출 — asyncio.run(main())이 이벤트 루프를 생성하고 main()을 실행
> Jupyter Notebook 환경이라 await asyncio.gather(...) 를 최상위에서 이미 사용 중
- 세 task 동시 등록 — asyncio.gather()가 A(3초), B(10초), C(2초)를 동시에 시작
- 각 task 실행 — await asyncio.sleep() 구간마다 이벤트 루프가 다른 task로 전환

In [11]:
# ── await의 의미를 이해하는 간단한 예시 ──
# ask("A", 3) 호출 시 → 함수 정의의 name, delay 파라미터에 순서대로 매핑됩니다.
async def task(name, delay):
    print(f"[{name}] 시작")
    await asyncio.sleep(delay)  # 'delay'초 동안 기다리는 동안 다른 task가 실행됨
    print(f"[{name}] 완료 ({delay}초 후)")

print("=== 비동기: 세 작업을 동시에 실행 ===")
t1 = time.time()

# wait asyncio.sleep(delay)를 만나는 순간, 해당 함수는 제어권을 이벤트 루프에 넘기고 "잠시 대기" 상태가 됩니다.
# 프로그램이 멈추는 것이 아니라, 다른 작업을 할 수 있도록 비켜주는 것입니다.
await asyncio.gather(
    task("A", 3), # name="A", delay=3
    task("B", 1), # name="B", delay=1
    task("C", 2),
    task("D", 3),
    task("E", 1),
    task("F", 2),
    task("G", 3),
    task("H", 1),
    task("I", 2),
)
t2 = time.time()
print(f"\n총 소요 시간: {t2 - t1:.1f}초 (순차라면 18초 소요됐을 것)")

=== 비동기: 세 작업을 동시에 실행 ===
[A] 시작
[B] 시작
[C] 시작
[D] 시작
[E] 시작
[F] 시작
[G] 시작
[H] 시작
[I] 시작
[B] 완료 (1초 후)
[E] 완료 (1초 후)
[H] 완료 (1초 후)
[C] 완료 (2초 후)
[F] 완료 (2초 후)
[I] 완료 (2초 후)
[A] 완료 (3초 후)
[D] 완료 (3초 후)
[G] 완료 (3초 후)

총 소요 시간: 3.0초 (순차라면 18초 소요됐을 것)


| 방식 | 계산 방식 | 예상 소요 시간 |
|------|----------|----------------|
| 순차 실행 (Synchronous) | 3 + 1 + 2 + 3 + 1 + 2 + 3 + 1 + 2 | 18초 |
| 비동기 실행 (Asynchronous) | max(3, 1, 2, 3, 1, 2, 3, 1, 2) | 3초 |

---
## **5. aiohttp로 웹 페이지 내용 가져오기**

`requests.get()` ↔ `async with session.get()` 의 대응 관계를 확인하세요.

동기 방식 (fetch_sync)

> requests.get()으로 URL에 요청을 보내고, 응답이 올 때까지 기다린 후 다음 코드를 실행합니다. 간단하지만, 여러 URL을 처리할 때 순서대로 하나씩 기다려야 해서 느립니다.

비동기 방식 (fetch_async)

> aiohttp를 사용해 요청을 보내고, 응답을 기다리는 동안 다른 작업을 병행할 수 있습니다. 여러 URL을 동시에 처리할 때 훨씬 빠릅니다.

In [12]:
# 동기 방식
def fetch_sync(url):
    response = requests.get(url, verify=False) # SSL 인증서 검증 비활성화
    return response.text

# 비동기 방식
async def fetch_async(url):
    # aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False))를 사용하여 SSL 인증서 검증 비활성화
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async with session.get(url, ssl=False) as response: # 개별 요청에서도 ssl=False 설정 (선택사항이지만 일관성을 위해)
            return await response.text()


url = 'https://example.com'

# 동기 실행
html_sync = fetch_sync(url)
print("[동기] 결과 (앞 200자):")
print(html_sync[:200])

print()

# 비동기 실행
# 경고메시지 때문에 다르게 보임. 내용은 동일함
html_async = await fetch_async(url)  # 일반 스크립트: asyncio.run(fetch_async(url))
print("[비동기] 결과 (앞 200자):")
print(html_async[:200])

[동기] 결과 (앞 200자):
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-famil

[비동기] 결과 (앞 200자):
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-famil


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'example.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
